[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/08.%20The%20Quay%20Crane%20Assignment%20Problem/P8-Tier-2_executed.ipynb)

---


# 8. The Quay Crane Assignment Problem
## Tier 2 — Beam Search

### Goal
This notebook uses beam search to assign quay cranes to vessels. A state represents a partial assignment. At each level, the search expands each retained state by considering feasible crane groups for the next vessel, scores the resulting states with a heuristic, and keeps at most a fixed number of the most promising states.

The example has four vessels and six cranes. The beam width is three. The input data are vessel workloads, priority values, minimum and maximum crane requirements, and a productivity rate for each vessel–crane pair. The output is a crane assignment and the estimated completion time for each vessel.

### Scope and modeling assumptions
- Crane productivity rates are deterministic estimates, expressed in TEU per hour.
- Cranes are indivisible resources: a crane can be assigned to at most one vessel.
- Each vessel must receive between its stated minimum and maximum number of cranes.
- The processing order is descending vessel priority. Priority determines the search order; it is not an additional weight in the completion-time score.
- When a vessel has \(k\) cranes, each crane's rate is reduced by the same multiplicative interference factor raised to \(k-1\). This is a simplified representation of interference.
- The reported aggregate is the sum of estimated vessel completion times. The model does not include vessel arrival times, berth positions, crane travel or adjacency constraints, uncertain productivity, or a shared scheduling timeline.
- Beam search is a heuristic. Its result depends on the heuristic score and beam width; it is not guaranteed to be globally optimal and has no general optimality bound.

### Search outline
1. Order vessels by priority.
2. Score partial assignments using estimated completion times, a rough estimate for unassigned work, and a crane-count balancing penalty.
3. Expand states one vessel at a time, discard states that cannot meet the remaining minimum crane requirements, and retain at most the beam width of lowest-scoring states.
4. Inspect the retained states and validate the final assignment.


## Why use beam search here?

A search that retains every partial assignment can grow rapidly: each vessel may have many possible crane groups, and choices at one level combine with choices at later levels. Beam search limits the number of retained states at each level. This makes the search more manageable, but pruning can discard a partial assignment that would have led to a better complete assignment.

The beam width controls this trade-off. A wider beam retains more candidates and generally uses more computation and memory; it does not guarantee a better answer for every instance. The method is useful when a bounded, heuristic search is appropriate, but its suitability for operational use depends on instance size, data quality, and required solution quality. No particular runtime or percentage of optimality is guaranteed by this example.


## Step 1: Problem setup and priority order

Each vessel has a workload in TEU, a priority, and a permitted range of crane counts. The productivity matrix has one row per vessel and one column per crane. Its entry in row \(i\), column \(c\) is the estimated productivity of crane \(c\) when serving vessel \(i\).

The search processes higher-priority vessels first. Crane indices in the code start at zero; printed crane labels start at one.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations
from typing import Dict, List, Tuple


class Vessel:
    def __init__(
        self,
        idx: int,
        workload: int,
        priority: int,
        min_cranes: int,
        max_cranes: int,
        name: str,
    ):
        self.idx = idx
        self.workload = workload  # Workload in TEU.
        self.priority = priority  # A higher value means higher search priority.
        self.min_cranes = min_cranes
        self.max_cranes = max_cranes
        self.name = name


# The vessel data define workloads, priorities, and feasible crane-count ranges.
vessels = [
    Vessel(0, 1000, 8, 1, 3, "V1"),
    Vessel(1, 1500, 6, 2, 4, "V2"),
    Vessel(2, 800, 9, 1, 2, "V3"),
    Vessel(3, 1200, 7, 1, 3, "V4"),
]

# Rows correspond to vessels; columns correspond to cranes. Rates are in TEU/hour.
productivity_rates = np.array(
    [
        [25, 27, 24, 26, 23, 25],  # V1
        [23, 25, 26, 24, 22, 24],  # V2
        [26, 24, 25, 27, 25, 26],  # V3
        [24, 26, 23, 25, 26, 24],  # V4
    ],
    dtype=float,
)

# Each additional crane applies one more factor of this interference multiplier.
interference_factor = 0.08

# At each level, retain no more than this many partial assignments.
beam_width = 3

assert productivity_rates.shape == (len(vessels), 6)
assert 0 <= interference_factor < 1
assert beam_width > 0
assert all(v.min_cranes <= v.max_cranes for v in vessels)
assert sum(v.min_cranes for v in vessels) <= productivity_rates.shape[1]

print("Problem setup:")
print(
    f"Vessels: {len(vessels)}, "
    f"cranes: {productivity_rates.shape[1]}, "
    f"beam width: {beam_width}"
)
print("\nVessel details:")
for vessel in vessels:
    print(
        f"  {vessel.name}: {vessel.workload} TEU, "
        f"priority {vessel.priority}, "
        f"cranes {vessel.min_cranes}-{vessel.max_cranes}"
    )

vessels_sorted = sorted(vessels, key=lambda vessel: vessel.priority, reverse=True)
print(
    "\nVessel processing order by priority: "
    f"{[vessel.name for vessel in vessels_sorted]}"
)

## Step 2: Heuristic evaluation

A partial assignment does not yet specify what will happen to every vessel. The heuristic gives each partial assignment a score so the search can decide which candidates to retain. Lower scores are preferred. This score guides the search; it is not an optimality certificate.

For a vessel \(v\) assigned a nonempty crane set \(C_v\), the estimated completion time is

$$
T_v(C_v) = \frac{w_v}{(1-\alpha)^{|C_v|-1}\sum_{c\in C_v} r_{vc}}.
$$

Here, \(w_v\) is the vessel workload in TEU, \(r_{vc}\) is the productivity rate of crane \(c\) for vessel \(v\) in TEU per hour, and α is the interference factor. The denominator is effective productivity in TEU per hour, so \(T_v\) is in hours. With one crane, the interference multiplier is one. With more cranes, the same multiplier is applied to each crane's rate, as specified by this model.

Let \(A\) be the vessels already assigned, \(R\) the remaining vessels, and \(C\) the available cranes. The heuristic used here is

$$
h(A,R,C) = \sum_{v\in A} T_v(C_v)
+ 0.8\,\frac{\sum_{v\in R} w_v}{|C|\,\bar r_C}
+ 2\,\operatorname{std}\bigl(|C_v|:v\in A\bigr),
$$

when remaining work and available cranes are both present. The estimated remaining time uses κr r_C, the mean productivity over all vessel rows and available-crane columns. The factor 0.8 discounts this rough estimate because future assignments are not yet known. The final term penalizes variation in the number of cranes assigned among vessels already in the partial assignment. If no vessels have yet been assigned, this balancing term is zero. If no work remains, the remaining-work term is zero.

In the code, `calculate_completion_time_heuristic` computes \(T_v\), and `heuristic_evaluation` computes \(h\). Both receive the data they use as arguments, making their inputs explicit.

In [ ]:
def calculate_completion_time_heuristic(
    vessel: Vessel,
    assigned_cranes: List[int],
    productivity_rates: np.ndarray,
    interference_factor: float,
) -> float:
    """Estimate a vessel's completion time in hours for its assigned cranes."""
    if not assigned_cranes:
        return float("inf")

    rates = productivity_rates[vessel.idx, assigned_cranes]
    num_cranes = len(assigned_cranes)
    interference_multiplier = (1 - interference_factor) ** (num_cranes - 1)
    effective_productivity = float(np.sum(rates) * interference_multiplier)

    if effective_productivity <= 0:
        return float("inf")

    return vessel.workload / effective_productivity


def heuristic_evaluation(
    partial_assignment: Dict[int, List[int]],
    remaining_vessels: List[Vessel],
    available_cranes: List[int],
    vessels: List[Vessel],
    productivity_rates: np.ndarray,
    interference_factor: float,
) -> float:
    """Score a partial assignment; lower heuristic scores are preferred."""
    vessels_by_idx = {vessel.idx: vessel for vessel in vessels}
    score = 0.0

    # Completed estimates for assigned vessels form the known part of the score.
    for vessel_idx, cranes in partial_assignment.items():
        vessel = vessels_by_idx[vessel_idx]
        score += calculate_completion_time_heuristic(
            vessel,
            cranes,
            productivity_rates,
            interference_factor,
        )

    # This coarse estimate gives the search information about work not yet assigned.
    if remaining_vessels and available_cranes:
        remaining_workload = sum(vessel.workload for vessel in remaining_vessels)
        average_productivity = float(np.mean(productivity_rates[:, available_cranes]))
        if average_productivity > 0:
            estimated_remaining_time = remaining_workload / (
                len(available_cranes) * average_productivity
            )
            score += 0.8 * estimated_remaining_time
        else:
            return float("inf")

    # Balancing crane counts discourages partial assignments that concentrate resources.
    crane_counts = [len(cranes) for cranes in partial_assignment.values()]
    if crane_counts:
        score += 2 * float(np.std(crane_counts))

    return score


# These example partial assignments illustrate how the heuristic combines its terms.
test_assignment = {0: [0, 1], 1: [2, 3]}
remaining_vessels_test = [vessel for vessel in vessels if vessel.idx in {2, 3}]
available_cranes_test = [4, 5]

heuristic_score = heuristic_evaluation(
    test_assignment,
    remaining_vessels_test,
    available_cranes_test,
    vessels,
    productivity_rates,
    interference_factor,
)
print(f"Example heuristic score: {heuristic_score:.3f}")

bad_assignment = {0: [0], 1: [1, 2, 3, 4]}
bad_score = heuristic_evaluation(
    bad_assignment,
    remaining_vessels_test,
    [5],
    vessels,
    productivity_rates,
    interference_factor,
)
print(f"Second example heuristic score: {bad_score:.3f}")
print(
    "The scores rank these two partial assignments under the stated heuristic; "
    "they are not proof of which leads to the best complete assignment."
)

## Step 3: Beam search

A **state** is a record of the assignments made so far, the cranes already used, the vessels still to process, and the state's heuristic score. At each level, the current vessel is assigned every feasible crane group in turn. Candidate states are scored, sorted, and truncated to the beam width.

The minimum-crane feasibility check is essential: after an assignment, the number of unused cranes must be at least the sum of the minimum requirements of all remaining vessels. This does not choose a different search method; it prevents the beam from retaining partial states that cannot possibly be completed.

If \(n\) is the number of vessels, \(B\) is the beam width, and \(Q\) is the maximum number of crane groups generated from one state at one level, the search generates at most about \(nBQ\) candidates, in addition to the cost of scoring and sorting them. The temporary candidate list can contain up to \(BQ\) states at a level. These bounds describe the beam search implementation; \(Q\) depends on the number of cranes and the permitted crane-count ranges. By contrast, retaining every partial assignment can produce a combinatorial growth in the number of states.

The state and transition logic below implement the priority-ordered, level-by-level search. The final state is the lowest-heuristic-score state retained after all vessels have been processed.

In [ ]:
class BeamSearchState:
    def __init__(
        self,
        assignment: Dict[int, List[int]],
        used_cranes: set,
        remaining_vessels: List[Vessel],
        heuristic_score: float,
    ):
        self.assignment = assignment.copy()
        self.used_cranes = used_cranes.copy()
        self.remaining_vessels = remaining_vessels.copy()
        self.heuristic_score = heuristic_score

    def __lt__(self, other):
        # A lower score should be ordered first if states are compared directly.
        return self.heuristic_score < other.heuristic_score


def generate_crane_combinations(
    available_cranes: List[int],
    min_cranes: int,
    max_cranes: int,
) -> List[List[int]]:
    """Return every available-crane group within the vessel's size limits."""
    combinations_list = []
    upper_limit = min(max_cranes, len(available_cranes))

    for num_cranes in range(min_cranes, upper_limit + 1):
        for crane_group in combinations(available_cranes, num_cranes):
            combinations_list.append(list(crane_group))

    return combinations_list


def beam_search(
    vessels: List[Vessel],
    beam_width: int,
    productivity_rates: np.ndarray,
    interference_factor: float,
) -> Tuple[Dict[int, List[int]], List[List[BeamSearchState]]]:
    """Assign cranes with priority-ordered, level-by-level beam search."""
    if beam_width <= 0:
        raise ValueError("beam_width must be a positive integer")
    if productivity_rates.ndim != 2 or productivity_rates.shape[0] != len(vessels):
        raise ValueError("productivity_rates must have one row per vessel")

    vessels_sorted = sorted(vessels, key=lambda vessel: vessel.priority, reverse=True)
    num_cranes = productivity_rates.shape[1]

    initial_state = BeamSearchState(
        assignment={},
        used_cranes=set(),
        remaining_vessels=vessels_sorted,
        heuristic_score=0.0,
    )
    current_beam = [initial_state]
    all_states_history = [current_beam.copy()]

    print(
        f"Starting beam search with {len(vessels_sorted)} vessels "
        f"and beam width {beam_width}"
    )

    for level, current_vessel in enumerate(vessels_sorted, start=1):
        print(
            f"\nLevel {level}: processing {current_vessel.name} "
            f"(priority {current_vessel.priority})"
        )
        new_states = []

        for state in current_beam:
            available_cranes = [
                crane
                for crane in range(num_cranes)
                if crane not in state.used_cranes
            ]
            crane_groups = generate_crane_combinations(
                available_cranes,
                current_vessel.min_cranes,
                current_vessel.max_cranes,
            )

            for cranes in crane_groups:
                new_assignment = state.assignment.copy()
                new_assignment[current_vessel.idx] = cranes

                new_used_cranes = state.used_cranes.copy()
                new_used_cranes.update(cranes)

                new_remaining = [
                    vessel
                    for vessel in state.remaining_vessels
                    if vessel.idx != current_vessel.idx
                ]
                remaining_cranes = [
                    crane
                    for crane in range(num_cranes)
                    if crane not in new_used_cranes
                ]

                # Preserve enough cranes to satisfy every remaining minimum requirement.
                required_cranes = sum(
                    vessel.min_cranes for vessel in new_remaining
                )
                if len(remaining_cranes) < required_cranes:
                    continue

                score = heuristic_evaluation(
                    new_assignment,
                    new_remaining,
                    remaining_cranes,
                    vessels,
                    productivity_rates,
                    interference_factor,
                )
                new_states.append(
                    BeamSearchState(
                        new_assignment,
                        new_used_cranes,
                        new_remaining,
                        score,
                    )
                )

        new_states.sort(key=lambda state: state.heuristic_score)
        current_beam = new_states[:beam_width]
        all_states_history.append(current_beam.copy())

        print(
            f"  Feasible candidates: {len(new_states)}; "
            f"retained states: {len(current_beam)}"
        )
        for state_number, state in enumerate(current_beam, start=1):
            print(
                f"    State {state_number}: "
                f"h={state.heuristic_score:.3f}, "
                f"assignment={state.assignment}"
            )

        if not current_beam:
            print("No feasible states remain at this level.")
            return {}, all_states_history

    best_state = min(current_beam, key=lambda state: state.heuristic_score)
    return best_state.assignment, all_states_history


best_assignment, search_history = beam_search(
    vessels,
    beam_width,
    productivity_rates,
    interference_factor,
)

print("\nFinal retained assignment:")
total_completion_time = 0.0
for vessel in vessels_sorted:
    cranes = best_assignment.get(vessel.idx, [])
    if not cranes:
        continue
    completion_time = calculate_completion_time_heuristic(
        vessel,
        cranes,
        productivity_rates,
        interference_factor,
    )
    total_completion_time += completion_time
    crane_names = [f"C{crane + 1}" for crane in cranes]
    print(
        f"  {vessel.name}: {crane_names} "
        f"({len(cranes)} cranes) -> {completion_time:.2f} hours"
    )

print(f"Sum of estimated vessel completion times: {total_completion_time:.2f} hours")

## Step 4: Search analysis and visualization

The search history records the states retained after each vessel is processed. The table summarizes the best, worst, and mean heuristic scores among those retained states. These scores belong to partial assignments and include an estimate for work not yet assigned; they are not the final objective value and need not decrease at every level.

The plots show the retained heuristic scores, the number of states retained, the crane counts in the final assignment, and the estimated completion times by vessel. The beam width limits retained states at each level; it does not indicate how many complete assignments exist.

In [ ]:
search_stats = []
for level, states in enumerate(search_history):
    if level == 0 or not states:
        continue

    heuristic_scores = [state.heuristic_score for state in states]
    search_stats.append(
        {
            "Level": level,
            "States retained": len(states),
            "Best heuristic": min(heuristic_scores),
            "Worst heuristic": max(heuristic_scores),
            "Mean heuristic": float(np.mean(heuristic_scores)),
        }
    )

if search_stats:
    df_stats = pd.DataFrame(search_stats)
    print("Beam search statistics:")
    print(df_stats.round(3).to_string(index=False))

    levels = [row["Level"] for row in search_stats]
    best_scores = [row["Best heuristic"] for row in search_stats]
    mean_scores = [row["Mean heuristic"] for row in search_stats]
    retained_counts = [row["States retained"] for row in search_stats]

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))

    axes[0, 0].plot(levels, best_scores, "o-", label="Best retained state")
    axes[0, 0].plot(levels, mean_scores, "s--", label="Mean of retained states")
    axes[0, 0].set_xlabel("Search level (vessels processed)")
    axes[0, 0].set_ylabel("Heuristic score (hours plus penalty)")
    axes[0, 0].set_title("Heuristic scores by search level")
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].bar(levels, retained_counts, color="skyblue", edgecolor="black")
    axes[0, 1].set_xlabel("Search level (vessels processed)")
    axes[0, 1].set_ylabel("Number of retained states")
    axes[0, 1].set_title("Beam occupancy by level")
    axes[0, 1].grid(True, axis="y", alpha=0.3)

    if best_assignment:
        assigned_vessels = [
            vessel for vessel in vessels_sorted if vessel.idx in best_assignment
        ]
        vessel_names = [vessel.name for vessel in assigned_vessels]
        crane_counts = [
            len(best_assignment[vessel.idx]) for vessel in assigned_vessels
        ]
        completion_times = [
            calculate_completion_time_heuristic(
                vessel,
                best_assignment[vessel.idx],
                productivity_rates,
                interference_factor,
            )
            for vessel in assigned_vessels
        ]

        axes[1, 0].bar(
            vessel_names,
            crane_counts,
            color="lightgreen",
            edgecolor="black",
        )
        axes[1, 0].set_xlabel("Vessel")
        axes[1, 0].set_ylabel("Number of cranes assigned")
        axes[1, 0].set_title("Crane allocation in the retained assignment")
        axes[1, 0].grid(True, axis="y", alpha=0.3)

        axes[1, 1].bar(
            vessel_names,
            completion_times,
            color="orange",
            edgecolor="black",
        )
        axes[1, 1].set_xlabel("Vessel")
        axes[1, 1].set_ylabel("Estimated completion time (hours)")
        axes[1, 1].set_title("Estimated completion time by vessel")
        axes[1, 1].grid(True, axis="y", alpha=0.3)

    fig.tight_layout()
    plt.show()

    retained_state_count = sum(len(states) for states in search_history)
    print(
        "\nRetained-state count includes the initial empty state: "
        f"{retained_state_count}"
    )
    print(
        "The candidate-generation effort depends on the number of feasible "
        "crane groups considered from each retained state."
    )
else:
    print("No completed search levels are available for analysis.")

## Step 5: Feasibility and result audit

A useful result must satisfy the assignment constraints, not merely have a low heuristic score. The checks below verify that every vessel is assigned, each crane is used at most once, and each vessel receives a permitted number of cranes. They also recompute the completion-time estimates and confirm that their sum matches the reported aggregate, within a floating-point tolerance.

These checks validate the output of the beam search; they do not replace its search procedure.

In [ ]:
if not best_assignment:
    raise RuntimeError("Beam search did not return a complete assignment.")

assert set(best_assignment) == {vessel.idx for vessel in vessels}

assigned_cranes_flat = [
    crane
    for cranes in best_assignment.values()
    for crane in cranes
]
assert len(assigned_cranes_flat) == len(set(assigned_cranes_flat))
assert all(
    0 <= crane < productivity_rates.shape[1]
    for crane in assigned_cranes_flat
)

audit_rows = []
recomputed_total = 0.0
for vessel in vessels_sorted:
    cranes = best_assignment[vessel.idx]
    assert vessel.min_cranes <= len(cranes) <= vessel.max_cranes

    completion_time = calculate_completion_time_heuristic(
        vessel,
        cranes,
        productivity_rates,
        interference_factor,
    )
    recomputed_total += completion_time
    audit_rows.append(
        {
            "Vessel": vessel.name,
            "Assigned cranes": ", ".join(
                f"C{crane + 1}" for crane in cranes
            ),
            "Crane count": len(cranes),
            "Estimated completion time (hours)": completion_time,
        }
    )

assert np.isclose(
    recomputed_total,
    total_completion_time,
    rtol=1e-9,
    atol=1e-9,
)

df_assignment = pd.DataFrame(audit_rows)
print("Validated crane assignment:")
print(
    df_assignment.to_string(
        index=False,
        formatters={
            "Estimated completion time (hours)": "{:.2f}".format
        },
    )
)
print(
    "\nSum of estimated vessel completion times: "
    f"{recomputed_total:.2f} hours"
)
print("All assignment and arithmetic checks passed.")

## Results interpretation and key insights

The assignment table reports which cranes serve each vessel and the completion-time estimate implied by the productivity and interference model. The sum is the aggregate used for reporting in this notebook; it is not a vessel makespan or a complete port schedule.

The search history shows how the beam width limits the number of partial assignments retained at each level. The heuristic combines known completion-time estimates, a coarse estimate for unassigned work, and a balancing penalty. Because this score is approximate, a low-scoring partial state is promising according to the heuristic, not guaranteed to lead to the best complete assignment.

The minimum-crane feasibility check ensures that retained states have enough unused cranes to satisfy the remaining minimum requirements. It does not establish optimality. Increasing the beam width allows more states to survive pruning, at the cost of additional computation and memory; the resulting assignment can still depend on the heuristic and on the input data.

### Further study
Natural extensions within this modeling framework include adding operational constraints, refining the productivity estimates, or testing how a chosen beam width affects search effort and assignment quality on additional instances. Any such study should state its data and assumptions and distinguish heuristic solution quality from a proven optimum.